# Collection Équinoxe : hausse de loyer 2026

**Défi CodeML 2026 – JADCO « Clés en main ».**
- **Sections 1 à 6 :** le notebook de départ `starter.ipynb`, repris tel quel. Seules deux sorties qui affichaient des lignes CRM brutes ont été remplacées par des agrégats.
- **Sections 7 à 19 :** notre travail.

Les fichiers sources ne sont jamais modifiés : tout renommage et toute transformation se font ici.

### La question à laquelle on répond

> **Hausse 2026 = croissance annualisée du loyer *effectif* (`sRentEffective`, après toutes les concessions), à unité constante (chaque appartement comparé à son bail précédent, clé `hUnit` ⇔ `sPropCode + sUnitCode`), pour les baux qui débutent en 2026.**
> On calcule la médiane de chaque segment (province × renouvellement/relocation), puis on pondère les segments par leur poids.

Pourquoi cette définition :

| Option | Pourquoi on ne la retient pas comme chiffre principal |
|---|---|
| Médiane annuelle du portefeuille | Mesure surtout la composition (immeubles ajoutés en 2023-2024), pas le prix (section 11). |
| Même unité, loyer **contractuel** | Ignore les concessions, qui passent d'environ 3 % à 9 % du loyer entre 2023 et 2025. Elle surestime donc ce qui est encaissé. On la publie quand même comme **plafond** : c'est le bon chiffre pour fixer les loyers affichés. |
| Même unité, loyer **net des seuls rabais de loyer** | Défendable comme « prix de l'appartement », mais il ignore le stationnement et le casier offerts, qui sont aussi un revenu abandonné. On la garde comme **sensibilité** (sections 13 et 17). |
| Renouvellements seulement / relocations seulement | Utiles pour la tarification, mais aucune ne représente seule le budget de revenus. On les prévoit séparément puis on les combine. |
| **Même unité, loyer effectif, segments pondérés** ✅ | Pas d'effet de mix, reflète ce que le propriétaire encaisse, et se décompose en leviers pilotables : règles (TAL / Ontario), marché, concessions. |

**Plan** (*Définir → Mesurer → Tester → Défendre*) :

| § | Contenu | Critère du jury |
|---|---|---|
| 1-6 | Notebook de départ | - |
| 7-8 | Constantes, contrôle des colonnes, renommage | Qualité, analyse des données |
| 9 | Clé d'unité | Appariement |
| 10 | Concessions : PromoPay, reconstruction de `sRentEffective`, 3 mesures de prix | Concessions |
| 11 | Effet de mix chiffré (indice à composition fixe) | Effet de mix |
| 12 | Croissance à unité constante | Appariement |
| 13 | Contractuel vs effectif, effet du bail précédent | Concessions |
| 14 | Renouvellements vs relocations, Québec vs Ontario | Renouvellements |
| 15 | Données publiques et réconciliation | Données externes |
| 16 | Modèle, `estimate_2026()`, `backtest()` | Prévision et backtest |
| 17 | Prévision 2026, scénarios, intervalles, par immeuble et par chambres | Prévision, bonus |
| 18-19 | Résultat, hypothèses, limites, conformité, références | - |

## 1. Charger les données

## 1b. Lire les colonnes : champs `h` et champs `s`

Chaque colonne de ce jeu commence par **`h`** ou **`s`**. C'est la convention de Yardi, et
se tromper là-dessus est la façon la plus courante de corrompre une requête de gestion immobilière.

| préfixe | signification | exemple | à quoi ça sert |
|---|---|---|---|
| **`h`** | **handle** (identifiant) - une clé numérique | `hUnit`, `hProperty`, `hBuilding` | **les jointures**. Opaque, stable, unique. |
| **`s`** | **stored value** (valeur stockée) - ce qu'un humain lit | `sUnitCode`, `sRent`, `sBuilding` | **l'affichage et le regroupement**. |

Yardi conserve les deux moitiés d'une même information côte à côte - `hAgent` à côté de `sAgent`,
`hProperty` à côté de `sPropCode`. Elles ne sont **pas interchangeables** :

> **Joindre sur `h`. Étiqueter avec `s`.**

Ce n'est pas du pinaillage, c'est tout l'objet de la section 4 ci-dessous. `sUnitCode` vaut `"0203"` - et
trois appartements différents sur un même site portent cette même chaîne. `hUnit` est unique par
construction. Si vous associez une unité à elle-même du côté `s`, vous fusionnez des appartements sans le savoir;
si vous associez sur `hUnit`, c'est impossible.

**Deux mises en garde sur `s`, qui reflètent toutes deux le vrai comportement de Yardi, pas des particularités de ce jeu :**

1. **`s` ne veut pas dire « string ».** La table `UNIT` de Yardi stocke `SRENT` et `DSQFT` comme
   *valeurs numériques*. Ici, `sRent` et `sSqft` sont des nombres. Ne déduisez pas un type d'un préfixe -
   vérifiez-le.
2. **Le vrai Yardi subdivise davantage le côté `s`** - `dt*` pour les dates (`dtLeaseFrom`), `d*`
   pour les doubles (`dSqft`), `i*` pour les entiers, `b*` pour les indicateurs. Ce jeu les ramène tous
   à `s` pour garder l'attention sur la distinction `h` / `s`. Quand vous ouvrirez la vraie base de données, attendez-vous
   à un alphabet plus long.

### Les codes de frais

`sChargeCode` dans le fichier des concessions correspond au plan comptable de Yardi. Les codes semblent abîmés
parce que `sCode` est un `nchar(8)` - « Free other » devient `Freeothe`, « Referencing » devient
`Referenc`. Huit caractères, tronqués, avec une casse incohérente. C'est à ça que ressemblent les vrais.

| `sChargeCode` | `sChargeDesc` | ce que c'est réellement |
|---|---|---|
| `PromoPay` | Promo - Payable in the future | Un rabais promotionnel. **Lisez la mise en garde ci-dessous.** |
| `freerent` | Free rent | Loyer annulé pour une partie du bail. |
| `freepark` | Free Parking | Frais de stationnement annulés. Un avantage *accessoire*, pas du loyer. |
| `freelock` | Free Locker | Casier de rangement offert. Accessoire. |
| `freeappl` | Free Appliances | Frais d'électroménagers annulés. Accessoire. |
| `Freeothe` | Free other | Tout ce qui n'est pas catégorisé. À traiter avec méfiance. |
| `Indem` | Client indemnity | Un paiement versé au résident, par exemple pour régler un différend. |
| `Referenc` | Referencing program | Un crédit de référencement. |

**Celui qu'il faut examiner avant de s'y fier :** `PromoPay` est décrit comme un frais
mensuel. Ce n'en est pas un. Il est très majoritairement inscrit sur un seul mois, pour un montant proche d'un mois
de loyer complet. Prenez la description au pied de la lettre et vous conclurez que les résidents habitent
presque gratuitement. Comparez `sMonths` et la taille de `sAmount` à `sRent` avant de le modéliser -
et décidez vous-même comment l'étaler sur la durée du bail.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

listings   = pd.read_csv("equinoxe_listings.csv")
leases     = pd.read_csv("equinoxe_lease_history.csv")
concessions= pd.read_csv("equinoxe_concessions.csv")
asking     = pd.read_csv("equinoxe_asking_history.csv")

for name, df in [("listings", listings), ("leases", leases),
                 ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

In [ ]:
# Convertir les dates une seule fois, dès le départ.
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    if col in leases.columns:
        leases[col] = pd.to_datetime(leases[col], errors="coerce")

leases["year"] = leases.sLeaseFrom.dt.year
asking["date"] = pd.to_datetime(asking.sMonth + "-01")

leases.dtypes.rename("type").to_frame()   # schéma seulement : aucune ligne CRM affichée

## 2. Ce que vous avez sous les yeux

Six immeubles répartis entre Laval, Mont-Royal, Ottawa et Pointe-Claire. Tout est en date du **2025-12-31**;
il n'y a pas de 2026 dans ce jeu, parce que 2026 est justement ce que vous prédisez.

Cinq sont au Québec et un, The Met, est en Ontario. Ce n'est pas
un détail : les deux provinces encadrent les hausses de loyer selon des régimes différents, et
les traiter comme un seul portefeuille est un choix de modélisation que vous devrez défendre.

**Une chose à bien comprendre avant de commencer.** Il s'agit d'un *extrait publié*, pas du
portefeuille. Il contient une part fixe des unités de chaque immeuble et de chaque type de chambre - la
répartition est représentative, les volumes ne le sont pas. Donc tout ce qui nécessite un dénominateur d'unités
réelles est hors de portée ici : **l'occupation, l'inoccupation, l'absorption et le rôle de loyers total du portefeuille
ne peuvent pas être calculés à partir de ce jeu, et un chiffre que vous en tireriez serait un artefact de
l'extrait plutôt qu'un fait sur l'entreprise.** C'est la *variation* des loyers que ces données permettent de mesurer,
et c'est la variation des loyers qu'on vous demande.

In [ ]:
print(listings.groupby(["sState", "sCity", "sBuilding"]).size().to_string())
print()
print(listings.groupby(["sBeds", "sBaths"])
      .agg(units=("sRent", "size"),
           median_rent=("sRent", "median"),
           median_sqft=("sSqft", "median"))
      .to_string())

## 3. L'approche évidente

Le premier réflexe naturel : le loyer médian par année.

In [ ]:
naive = (leases[leases.year.between(2019, 2025)]
         .groupby("year")
         .agg(leases=("sRent", "size"), median_rent=("sRent", "median")))
naive["yoy_pct"] = (naive.median_rent.pct_change() * 100).round(2)
naive

Selon ce tableau, les loyers auraient augmenté de **10,8 % en 2023**. Ce n'est pas le cas.

La médiane bouge dès que la *composition* bouge, et la composition a beaucoup bougé. Plusieurs
immeubles ont été mis en service en 2023 et 2024. Regardez ce qui se passe sous la
médiane :

In [ ]:
window = leases[leases.year.between(2019, 2025)]
mix = window.groupby("year").agg(
    leases=("sRent", "size"),
    median_sqft=("sSqft", "median"),
    pct_2bed_plus=("sBeds", lambda s: (s >= 2).mean() * 100),
    buildings=("sBuilding", "nunique"))
mix["rent_per_sqft"] = (window.assign(psf=window.sRent / window.sSqft)
                        .groupby("year").psf.median())
mix.round(2)

In [ ]:
# À partir de quand chaque sBuilding a-t-il commencé à générer des baux?
print(window.groupby("sBuilding").year.min().sort_values().to_string())
print()
# Et combien demandent-ils? (baux de 2024)
print(window[window.year == 2024].groupby("sBuilding")
      .agg(leases=("sRent", "size"), median_rent=("sRent", "median"))
      .sort_values("median_rent").to_string())

Les unités louées en 2024 sont **plus petites** que celles louées en 2021 : la superficie
médiane est passée de 1 175 à 1 045 pi², et pourtant le loyer médian a fortement augmenté. C'est
le contraire de ce que la taille seule laisserait prévoir. La part des baux de 2 chambres et plus
est passée de 78 % à 62 % sur la même période.

Ce qui s'est réellement passé : Le Carlyle, The Met et Westpark ont été mis en service en 2023 et 2024. Ce sont des
immeubles haut de gamme, qui louent des unités plus petites, davantage axées sur les 1 chambre, à des tarifs au pied carré
bien plus élevés - la médiane 2024 de The Met est de 2 752 $, contre 1 672 $ pour Lévesque.
Le loyer au pied carré est passé de 1,71 $ à 2,20 $ en deux ans, et presque rien de
cela ne vient d'un locataire existant qui paie plus.


## 4. La méthode qui fonctionne : comparer une unité à elle-même

Associez les baux consécutifs de chaque unité et annualisez la variation. Même appartement,
même immeuble, tout pareil : ce qui reste, c'est le prix.

**La seule chose à ne pas rater :** associez sur `sPropCode` + `sUnitCode`.
Pas sur `sSite` + `sUnitCode`. Plusieurs sites regroupent plus d'un code de propriété avec
des numéros d'unité qui se chevauchent, et **248 des 1 061 unités entrent en collision** si vous utilisez
le site comme clé. Exécutez la cellule suivante pour le constater, puis utilisez `sPropCode` + `sUnitCode`, ou
la colonne `sLink`, qui est construite sur la clé sûre.

In [ ]:
# Pourquoi sSite + sUnitCode n'est pas une clé : un site, un numéro d'unité, plusieurs appartements.
# (hUnit est l'identifiant qui EST unique - voir le dictionnaire de données plus haut.)
dupes = listings[listings.duplicated(["sSite", "sUnitCode"], keep=False)]
print(f"{len(dupes):,} unités sur {len(listings):,} entrent en collision sur sSite + sUnitCode")
print()
# (sortie réduite : seuls les codes de propriété sont affichés, pas les lignes CRM)
print("saint-elzear / 0203 ->", listings[(listings.sSite == "saint-elzear") & (listings.sUnitCode == "0203")]
      .sPropCode.tolist())

In [ ]:
UNIT_KEY = ["sPropCode", "sUnitCode"]      # <-- pas ["sSite", "sUnitCode"]

def same_unit_growth(df, price_col="sRent", date_col="sLeaseFrom",
                     min_gap=0.5, max_gap=2.5):
    """Variation annualisée du loyer, en comparant chaque unité à son propre bail précédent.

    Conserve les écarts entre `min_gap` et `max_gap` années : plus court signifie
    généralement le même bail enregistré deux fois, plus long est trop ancien pour une comparaison équivalente.
    """
    d = df.sort_values(UNIT_KEY + [date_col]).copy()
    d["prev_price"] = d.groupby(UNIT_KEY)[price_col].shift(1)
    d["prev_date"]  = d.groupby(UNIT_KEY)[date_col].shift(1)
    d = d.dropna(subset=["prev_price", "prev_date"])

    d["gap_years"] = (d[date_col] - d.prev_date).dt.days / 365.25
    d = d[d.gap_years.between(min_gap, max_gap, inclusive="neither")]
    d = d[(d.prev_price > 0) & (d[price_col] > 0)]

    d["growth_pct"] = ((d[price_col] / d.prev_price) ** (1 / d.gap_years) - 1) * 100
    return d

pairs = same_unit_growth(leases)
print(f"{len(pairs):,} paires utilisables sur {pairs.groupby(UNIT_KEY).ngroups:,} unités")

honest = (pairs[pairs.year.between(2021, 2025)]
          .groupby("year")
          .growth_pct.agg(pairs="size", median="median").round(2))
honest

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
yrs = range(2021, 2026)
ax.plot(yrs, naive.loc[yrs, "yoy_pct"], "o--", label="naïf : médiane de tous les baux", lw=2)
ax.plot(yrs, honest.loc[yrs, "median"], "o-", label="même unité comparée à elle-même", lw=2)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("croissance annuelle des loyers (%)")
ax.set_title("La méthode naïve mesure surtout la composition")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Loyer contractuel n'est pas loyer perçu

`sRent` est ce que le bail indique. `sRentEffective` est ce qu'Équinoxe a réellement
perçu après les mois gratuits, le stationnement gratuit et les casiers gratuits. Les concessions sont maintenant
la norme, et elles progressent.

In [ ]:
conc = (leases[leases.year.between(2021, 2025)]
        .groupby("year")
        .agg(leases=("sRent", "size"),
             pct_with_concession=("sConcession", lambda s: s.mean() * 100),
             median_contracted=("sRent", "median"),
             median_effective=("sRentEffective", "median")))
conc["gap_pct"] = ((conc.median_effective / conc.median_contracted - 1) * 100)
conc.round(2)

In [ ]:
# La concession change-t-elle l'évolution, ou seulement le niveau?
eff = same_unit_growth(leases, price_col="sRentEffective")
compare = pd.DataFrame({
    "contracted": honest["median"],
    "effective":  eff[eff.year.between(2021, 2025)].groupby("year").growth_pct.median(),
}).round(2)
compare

Ces deux colonnes évoluent de près jusqu'en 2023 - à moins
d'un point l'une de l'autre - puis elles divergent : en 2025, la croissance contractuelle est de
7,0 % alors que la croissance effective est de 3,6 %. L'écart de *niveau* du tableau précédent
se creuse tout au long de la période.

Elles répondent à des questions différentes, et l'année où elles se séparent est celle où les concessions
deviennent plus importantes plutôt que simplement plus fréquentes. Si vous pouvez expliquer ce que cette séparation signifie pour un
propriétaire - et laquelle des deux vous citeriez à un propriétaire - vous comprenez les données
mieux que la plupart des équipes.

## 6. Les renouvellements ne se comportent pas comme les relocations

`sRenewal = 1`, c'est un locataire en place qui renouvelle. `sRenewal = 0`, c'est une unité
qui se libère et qui est relouée au prix du marché. Au Québec, ces deux cas sont régis par
des règles différentes, et l'écart entre les deux est toute la raison pour laquelle un propriétaire s'intéresse
à ce chiffre. Remarquez que l'écart change de camp au cours de la période : les renouvellements mènent en 2022, les relocations en 2025.

In [ ]:
split = (pairs[pairs.year.between(2022, 2025)]
         .groupby(["year", "sRenewal"])
         .growth_pct.agg(pairs="size", median="median").round(2)
         .unstack("sRenewal"))
split.columns = [f"{a}_{'renewal' if b else 'turnover'}" for a, b in split.columns]
split

## 7. Configuration et constantes

Tous les paramètres sont nommés ici : aucun nombre « magique » dans la suite du notebook.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

# --- Fichiers -----------------------------------------------------------------
DATA_DIR = Path(".")
EXTERNAL_DATA_FILE = DATA_DIR / "donnees_externes.csv"   # données publiques, sources dans le fichier
MODEL_PARAMS_FILE = DATA_DIR / "model_params.json"        # paramètres du modèle (aucune donnée CRM)

# --- Horizon -------------------------------------------------------------------
AS_OF_DATE = pd.Timestamp("2025-12-31")   # date de l'extrait
FORECAST_YEAR = 2026
BACKTEST_YEARS = (2023, 2024, 2025)
ANALYSIS_START_YEAR = 2019                # premières années trop minces (< 100 baux)

# --- Appariement à unité constante ---------------------------------------------
JOIN_KEY = "hUnit"                         # handle Yardi : unique par construction
UNIT_KEY = ["sPropCode", "sUnitCode"]      # clé lisible équivalente (jamais sSite + sUnitCode)
WRONG_UNIT_KEY = ["sSite", "sUnitCode"]    # montrée uniquement pour illustrer le piège
MIN_GAP_YEARS = 0.5                        # plus court : même bail saisi deux fois / bail court atypique
MAX_GAP_YEARS = 2.5                        # plus long : unité vacante longtemps, comparaison non équivalente
DAYS_PER_YEAR = 365.25
PERCENT = 100

# --- Concessions -----------------------------------------------------------------
RENT_CONCESSION_CODES = {"PromoPay", "freerent"}                       # réduisent le loyer
ANCILLARY_CONCESSION_CODES = {"freepark", "freelock", "freeappl"}     # services accessoires
OTHER_CONCESSION_CODES = {"Freeothe", "Indem", "Referenc"}            # hors loyer (crédits, indemnités)
ONE_SHOT_MONTHS = 1                        # une concession sur 1 seul mois = montant forfaitaire
RECON_TOLERANCE_DOLLARS = 1.0              # écart accepté pour la reconstruction du loyer effectif
CONCESSION_START_TOLERANCE = pd.Timedelta(days=31)   # promo de signature inscrite avant l'emménagement

# --- Règles provinciales -------------------------------------------------------
QUEBEC, ONTARIO = "Quebec", "Ontario"
ONTARIO_EXEMPTION_DATE = pd.Timestamp("2018-11-15")   # 1re occupation après : hors ligne directrice
QC_SECTION_F_MAX_AGE_YEARS = 5                         # bail section F : immeuble de 5 ans ou moins

# --- Modèle ----------------------------------------------------------------------
GAP_WINDOW_YEARS = 3                       # années pour l'écart relocation - renouvellement
DISCOUNT_TREND_YEARS = 2                   # années pour la tendance du rabais (variation annuelle moyenne)
BASELINE_WINDOW_YEARS = 3                  # années du modèle de référence « moyenne 3 ans »
CONCESSION_DAMPING = {"persistance": 0.0, "tendance amortie": 0.5, "tendance pleine": 1.0}
PHI_GRID = np.round(np.arange(0, 1.51, 0.25), 2)   # grille de sensibilité (au-delà de 1 = accélération)
N_BOOTSTRAP = 500
BOOTSTRAP_CI = (2.5, 97.5)
RANDOM_SEED = 42

## 8. Charger les données et vérifier ce que les colonnes représentent

Convention Yardi : les colonnes **`h`** (handle) servent aux jointures, les colonnes **`s`** (valeur stockée) à l'affichage.
`sUnitCode` est lu comme texte pour conserver les zéros initiaux (`"0102"`).

In [ ]:
read_opts = dict(dtype={"sUnitCode": str})
listings_raw    = pd.read_csv(DATA_DIR / "equinoxe_listings.csv", **read_opts)
leases_raw      = pd.read_csv(DATA_DIR / "equinoxe_lease_history.csv", **read_opts)
concessions_raw = pd.read_csv(DATA_DIR / "equinoxe_concessions.csv", **read_opts)
asking_raw      = pd.read_csv(DATA_DIR / "equinoxe_asking_history.csv", **read_opts)

for name, df in [("listings", listings_raw), ("leases", leases_raw),
                 ("concessions", concessions_raw), ("asking", asking_raw)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

### 8a. Vérifications avant de renommer

On ne se fie pas aux descriptions : chaque hypothèse sur une colonne est testée.

In [ ]:
lease_dates = leases_raw.assign(**{c: pd.to_datetime(leases_raw[c]) for c in
                                   ["sSignDate", "sLeaseFrom", "sLeaseTo", "sAvailable"]})

checks = {
    "sAvailable == sLeaseFrom (même information)":
        (lease_dates.sAvailable == lease_dates.sLeaseFrom).mean(),
    "sConcession == 1  <=>  sRentEffective < sRent":
        ((lease_dates.sConcession == 1) == (lease_dates.sRentEffective < lease_dates.sRent)).mean(),
    "sRentEffective <= sRent":
        (lease_dates.sRentEffective <= lease_dates.sRent).mean(),
    "sSignDate <= sLeaseFrom":
        (lease_dates.sSignDate <= lease_dates.sLeaseFrom).mean(),
    "sTermSeq == 1  =>  sRenewal == 0":
        (lease_dates.loc[lease_dates.sTermSeq == 1, "sRenewal"] == 0).mean(),
    "sTermSeq >= 2 et sRenewal == 0 (relocation d'une unité déjà louée)":
        ((lease_dates.sTermSeq >= 2) & (lease_dates.sRenewal == 0)).mean(),
}
print(pd.Series(checks, name="part des baux").map("{:.1%}".format).to_string())
print("\nDélai signature -> début (jours) :")
print((lease_dates.sLeaseFrom - lease_dates.sSignDate).dt.days.describe().round(1).to_string())
print("\nsTermMonths :", lease_dates.sTermMonths.value_counts().head(5).to_dict())

**Ce que l'on en conclut :**
- `sAvailable` est **identique** à `sLeaseFrom` dans les baux. C'est la date d'emménagement, pas une date de disponibilité. On la renomme `move_in_date`.
- `sConcession` est un **indicateur** (0/1) : vaut 1 exactement quand le loyer effectif est sous le contractuel. Ce n'est pas un montant.
- `sTermSeq` est le **rang du bail dans l'historique de l'unité**, pas du locataire : un `sTermSeq ≥ 2` peut être une relocation. On utilise donc **`sRenewal`** (et non `sTermSeq`) pour séparer renouvellements et relocations.
- `sTermMonths` contient des durées fractionnaires (17,9 ; 23,9) : c'est une durée calculée, cohérente avec `sLeaseTo − sLeaseFrom`.
- Dans `equinoxe_listings.csv`, `sRent` n'est pas un loyer de bail : c'est le **loyer affiché courant** (à la date `sAsOf`) pour la prochaine disponibilité `sAvailable`.

### 8b. Renommage

On renomme selon ce que les données **représentent réellement**, puis on convertit les dates une seule fois.
Les clés Yardi (`hUnit`, `hProperty`, `sPropCode`, `sUnitCode`, `sSite`) gardent leur nom d'origine pour rester traçables.

In [ ]:
LEASE_RENAME = {
    "sRent": "rent_contract", "sRentEffective": "rent_effective", "sConcession": "has_concession",
    "sSignDate": "sign_date", "sLeaseFrom": "lease_start", "sLeaseTo": "lease_end",
    "sAvailable": "move_in_date", "sTermMonths": "term_months", "sTermSeq": "unit_lease_seq",
    "sRenewal": "is_renewal", "sLeaseTerm": "lease_term_label", "sBuilding": "building",
    "sState": "province", "sCity": "city", "sBeds": "beds", "sBaths": "baths", "sSqft": "sqft",
    "sFloor": "floor", "sUnitType": "unit_type", "sUnitSubtype": "unit_subtype",
}
CONCESSION_RENAME = {
    "sChargeCode": "charge_code", "sChargeDesc": "charge_desc", "sAmount": "amount_per_period",
    "sDateFrom": "date_from", "sDateTo": "date_to", "sMonths": "n_months",
    "sBuilding": "building", "sState": "province", "sBeds": "beds", "sSqft": "sqft",
}
ASKING_RENAME = {"sAskingRent": "asking_rent", "sMonth": "month", "sBuilding": "building",
                 "sState": "province", "sBeds": "beds", "sSqft": "sqft"}
LISTING_RENAME = {"sRent": "asking_rent_current", "sAsOf": "asking_as_of",
                  "sAvailable": "next_available_date", "sBuilding": "building",
                  "sState": "province", "sBeds": "beds", "sSqft": "sqft"}

leases = leases_raw.rename(columns=LEASE_RENAME)
for col in ["sign_date", "lease_start", "lease_end", "move_in_date"]:
    leases[col] = pd.to_datetime(leases[col])
leases["lease_year"] = leases.lease_start.dt.year
leases["discount_rate"] = 1 - leases.rent_effective / leases.rent_contract   # part du loyer cédée

concessions = concessions_raw.rename(columns=CONCESSION_RENAME)
for col in ["date_from", "date_to"]:
    concessions[col] = pd.to_datetime(concessions[col])

asking = asking_raw.rename(columns=ASKING_RENAME)
asking["month_date"] = pd.to_datetime(asking.month + "-01")

listings = listings_raw.rename(columns=LISTING_RENAME)
listings["next_available_date"] = pd.to_datetime(listings.next_available_date)

# Table de correspondance (on n'affiche pas de lignes CRM brutes : aucune donnée CRM dans les livrables)
pd.Series(LEASE_RENAME, name="nouveau nom").rename_axis("colonne Yardi").to_frame()

## 9. La clé d'unité

On fait les jointures sur `hUnit` (handle unique) et on vérifie qu'il est **strictement équivalent** à `sPropCode + sUnitCode`.
On montre aussi pourquoi `sSite + sUnitCode` est interdit.

In [ ]:
units_per_key = leases.groupby(UNIT_KEY)[JOIN_KEY].nunique()
keys_per_unit = leases.groupby(JOIN_KEY)[UNIT_KEY].apply(lambda d: len(d.drop_duplicates()))
print(f"hUnit distincts par (sPropCode, sUnitCode) : max = {units_per_key.max()}")
print(f"(sPropCode, sUnitCode) distincts par hUnit : max = {keys_per_unit.max()}")
key_counts = pd.Series({
    "hUnit": leases[JOIN_KEY].nunique(),
    "sPropCode + sUnitCode": leases[UNIT_KEY].drop_duplicates().shape[0],
    "sLink": leases.sLink.nunique(),
    "sSite + sUnitCode (piège)": leases[WRONG_UNIT_KEY].drop_duplicates().shape[0],
    "unités dans listings": len(listings)}, name="unités distinctes")
print(); print(key_counts.to_string())

collisions = listings[listings.duplicated(WRONG_UNIT_KEY, keep=False)]
print(f"\n{len(collisions):,} unités sur {len(listings):,} entrent en collision sur sSite + sUnitCode")
print(listings.groupby("sSite").sPropCode.unique().to_string())

`hUnit` ⇔ `sPropCode + sUnitCode` est une bijection : les deux clés donnent **exactement** les mêmes paires (vérifié plus bas par une assertion).
`sSite` regroupe plusieurs codes de propriété (ex. `saint-elzear` = `stelz1`, `stelz2`, `stelz3`) avec des numéros d'unité qui se recoupent.

## 10. Concessions : que représente vraiment `sAmount` ?

Hypothèse à tester : `sAmount` est un montant **par période**. Pour les codes récurrents (stationnement, casier…), la période est le mois.
Pour `PromoPay`, la « période » est un **seul mois** : c'est un rabais forfaitaire, pas un rabais mensuel.

In [ ]:
concessions["is_one_shot"] = concessions.n_months == ONE_SHOT_MONTHS
concessions["concession_total"] = concessions.amount_per_period * concessions.n_months  # négatif = crédit

summary = concessions.groupby("charge_code").agg(
    n=("amount_per_period", "size"),
    median_amount=("amount_per_period", "median"),
    median_months=("n_months", "median"),
    share_one_shot=("is_one_shot", "mean")).round(2)
print(summary.sort_values("n", ascending=False).to_string())

# La table des concessions n'a pas d'identifiant de bail. On joint sur hUnit (règle Yardi), puis on rattache
# chaque concession au bail de cette unité dont la période contient son début. On accepte une tolérance
# avant l'emménagement, car une promotion de signature peut être inscrite avant. Une concession -> un seul bail.
leases["lease_id"] = np.arange(len(leases))
candidates = concessions.reset_index(names="concession_id").merge(
    leases[[JOIN_KEY, "lease_id", "lease_start", "lease_end", "rent_contract", "rent_effective",
            "term_months", "lease_year"]], on=JOIN_KEY)
in_lease = ((candidates.date_from >= candidates.lease_start - CONCESSION_START_TOLERANCE)
            & (candidates.date_from <= candidates.lease_end))
candidates = candidates[in_lease].assign(
    distance_to_start=lambda d: (d.date_from - d.lease_start).abs())
concession_lease = (candidates.sort_values("distance_to_start")
                    .drop_duplicates("concession_id"))          # le bail le plus proche
promo = concession_lease[concession_lease.charge_code == "PromoPay"]
print(f"\n{len(concession_lease):,} concessions sur {len(concessions):,} rattachées à un bail du même hUnit")
print("PromoPay en mois de loyer (|montant| / loyer contractuel) :")
print((-promo.amount_per_period / promo.rent_contract).describe().round(2).to_string())

`PromoPay` est inscrit sur **un seul mois** dans ~88 % des cas, pour environ **0,8 mois de loyer** en médiane.
Lu comme un montant mensuel, il laisserait croire que les locataires habitent presque gratuitement. C'est en réalité **un mois gratuit (ou presque) offert une fois**, qu'il faut étaler sur la durée du bail.

### 10a. Reconstruire `sRentEffective` pour valider l'interprétation

Si l'interprétation est bonne, on doit retrouver :
`rent_effective ≈ rent_contract + (Σ concessions comprises dans le bail, en total) / term_months`.

On teste trois variantes :
- seulement les codes « loyer » (`PromoPay`, `freerent`) ;
- les codes loyer et les codes accessoires ;
- tous les codes.

In [ ]:
def concession_per_month(concession_codes):
    # Valeur mensuelle (négative) des concessions de ces codes, étalée sur la durée de chaque bail.
    subset = concession_lease[concession_lease.charge_code.isin(concession_codes)]
    total_by_lease = subset.groupby("lease_id").concession_total.sum()
    return leases.lease_id.map(total_by_lease).fillna(0) / leases.term_months


def reconstruct_effective(concession_codes):
    # Loyer effectif reconstruit : loyer contractuel + concessions totales étalées sur le bail.
    out = leases[["rent_contract", "rent_effective"]].copy()
    out["rent_effective_rebuilt"] = out.rent_contract + concession_per_month(concession_codes)
    out["error"] = out.rent_effective_rebuilt - out.rent_effective
    return out

CODE_VARIANTS = {
    "loyer seulement": RENT_CONCESSION_CODES,
    "loyer + accessoires": RENT_CONCESSION_CODES | ANCILLARY_CONCESSION_CODES,
    "tous les codes": RENT_CONCESSION_CODES | ANCILLARY_CONCESSION_CODES | OTHER_CONCESSION_CODES,
}
with_concession = leases.has_concession == 1
recon_table = {}
for label, codes in CODE_VARIANTS.items():
    errors = reconstruct_effective(codes).error.abs()
    recon_table[label] = {
        f"tous les baux à ±{RECON_TOLERANCE_DOLLARS:.0f} $": (errors <= RECON_TOLERANCE_DOLLARS).mean(),
        f"baux avec concession à ±{RECON_TOLERANCE_DOLLARS:.0f} $": (errors[with_concession] <= RECON_TOLERANCE_DOLLARS).mean(),
        "écart absolu médian, baux avec concession ($)": errors[with_concession].median()}
print(f"{len(concession_lease):,} concessions rattachées sur {len(concessions):,} ; "
      f"baux avec concession (has_concession = 1) : {with_concession.sum():,}")
pd.DataFrame(recon_table).round(3)

C'est la version **« tous les codes »** qui reproduit `sRentEffective` : à 1 $ près pour environ 9 baux sur 10. Sans les accessoires, il manque environ 70 $ par mois en médiane.
On retient donc :
- **`sRentEffective` = loyer contractuel − toutes les concessions du bail, étalées sur sa durée.** Cela comprend les mois gratuits (PromoPay, freerent), mais aussi le stationnement, le casier et les électroménagers gratuits. C'est cohérent avec la définition du notebook de départ.
- C'est donc un **revenu net par unité** : ce que le propriétaire encaisse réellement pour l'appartement et ses accessoires.
- Les écarts restants portent surtout sur des PromoPay que Yardi semble amortir autrement, ou sur des concessions qui chevauchent deux baux.

### 10b. Trois mesures de prix, et pourquoi on retient `sRentEffective`

Un stationnement offert est-il une baisse du **prix** de l'appartement ? On peut défendre les deux réponses. On construit donc une troisième mesure, `rent_net`, pour voir si le choix change la conclusion :
- **`rent_contract`** (`sRent`) : le loyer du bail. C'est le plafond.
- **`rent_net`** : le contractuel moins les seuls rabais **de loyer** (PromoPay, freerent), étalés sur le bail. C'est le prix de l'appartement.
- **`rent_effective`** (`sRentEffective`) : le contractuel moins **toutes** les concessions. C'est le revenu réellement encaissé, et c'est notre mesure principale.

Notre choix : **`rent_effective`**. Le budget de revenus porte sur ce qui est encaissé, et un stationnement offert est un revenu abandonné au même titre qu'un mois gratuit. C'est aussi la mesure que l'énoncé demande d'utiliser.
On garde `rent_net` comme analyse de sensibilité, et on la prévoit avec la même méthode à la section 17.

In [ ]:
leases["rent_concession_monthly"] = -concession_per_month(RENT_CONCESSION_CODES)          # positif = rabais
leases["ancillary_concession_monthly"] = -concession_per_month(ANCILLARY_CONCESSION_CODES)
leases["other_concession_monthly"] = -concession_per_month(OTHER_CONCESSION_CODES)
leases["rent_net"] = leases.rent_contract - leases.rent_concession_monthly
leases["net_discount_rate"] = leases.rent_concession_monthly / leases.rent_contract   # rabais de loyer seulement

concession_value = (leases[leases.lease_year >= ANALYSIS_START_YEAR].groupby("lease_year")
                    [["rent_contract", "rent_concession_monthly", "ancillary_concession_monthly", "other_concession_monthly"]].sum())
concession_share = (concession_value.iloc[:, 1:].div(concession_value.rent_contract, axis=0) * PERCENT)
concession_share.columns = ["rabais de loyer", "accessoires", "autres"]
ax = concession_share.plot.bar(stacked=True, figsize=(9, 4), color=["#6a3d9a", "#ff7f00", "#b2b2b2"], rot=0)
ax.set_title("Valeur des concessions, en % du loyer contractuel"); ax.set_ylabel("%"); ax.set_xlabel("")
ax.grid(alpha=0.3, axis="y"); plt.tight_layout(); plt.show()
concession_share.round(2)

In [ ]:
concession_by_year = (leases[leases.lease_year >= ANALYSIS_START_YEAR]
                      .groupby("lease_year")
                      .agg(leases=("rent_contract", "size"),
                           share_with_concession=("has_concession", "mean"),
                           mean_discount_rate=("discount_rate", "mean")))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(concession_by_year.index, concession_by_year.mean_discount_rate * PERCENT, color="#c0504d")
ax.set_ylabel("rabais moyen sur le loyer (%)")
ax.set_title("Les concessions deviennent plus importantes, pas seulement plus fréquentes")
for year, row in concession_by_year.iterrows():
    ax.annotate(f"{row.share_with_concession:.0%} des baux", (year, row.mean_discount_rate * PERCENT),
                ha="center", va="bottom", fontsize=8)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()
(concession_by_year.assign(share_with_concession=lambda d: d.share_with_concession * PERCENT,
                           mean_discount_rate=lambda d: d.mean_discount_rate * PERCENT)).round(2)

## 11. Effet de composition (mix) : pourquoi la médiane annuelle trompe

La médiane de tous les baux d'une année mélange deux choses : la variation des prix **et** le changement de composition du portefeuille.

In [ ]:
window = leases[leases.lease_year >= ANALYSIS_START_YEAR]
naive = window.groupby("lease_year").agg(leases=("rent_contract", "size"),
                                         median_rent=("rent_contract", "median"))
naive["naive_yoy_pct"] = naive.median_rent.pct_change() * PERCENT

mix = window.groupby("lease_year").agg(
    buildings=("building", "nunique"),
    median_sqft=("sqft", "median"),
    share_2bed_plus=("beds", lambda s: (s >= 2).mean() * PERCENT))
mix["median_rent_per_sqft"] = (window.rent_contract / window.sqft).groupby(window.lease_year).median()
first_year_by_building = leases.groupby("building").lease_year.min().sort_values()
print("Premier bail observé par immeuble :")
print(first_year_by_building.to_string())
naive.join(mix).round(2)

In [ ]:
leases_by_building_year = pd.crosstab(window.lease_year, window.building)
median_by_building = window.groupby("building").rent_contract.median().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
leases_by_building_year[median_by_building.index].plot(kind="bar", stacked=True, ax=axes[0], colormap="viridis")
axes[0].set_title("Baux par année : 3 immeubles haut de gamme arrivent en 2023-2024")
axes[0].set_xlabel(""); axes[0].set_ylabel("baux")
axes[1].barh(median_by_building.index, median_by_building.values, color="#4f81bd")
axes[1].set_title("Loyer contractuel médian par immeuble ($)")
plt.tight_layout(); plt.show()

Le Carlyle, The Met et Westpark entrent dans l'historique en 2023 et 2024. Leurs loyers sont bien plus élevés (The Met ≈ 2 780 $, Lévesque ≈ 1 525 $).
Leurs unités sont plus petites mais plus chères au pied carré. La médiane annuelle « monte » donc mécaniquement, même si aucun locataire existant ne paie plus.

### 11a. Chiffrer l'effet de mix : indice à composition fixe

Pour isoler le prix, on calcule un **indice de Laspeyres chaîné** :
- Pour chaque cellule *immeuble × nombre de chambres* présente deux années de suite, on calcule la variation de son loyer moyen.
- On pondère ces variations par le nombre de baux de l'année précédente.
- Un nouvel immeuble n'entre dans l'indice qu'à partir de sa deuxième année.

L'**effet de mix** = variation naïve − variation à composition fixe. On compare ensuite à la mesure à unité constante (section 12), qui est encore plus fine puisqu'elle compare chaque appartement à lui-même.

In [ ]:
MIX_CELL = ["building", "beds"]

def fixed_composition_growth_pct(lease_df, price_col="rent_contract"):
    # Variation annuelle (%) du loyer moyen à composition fixe (Laspeyres chaîné sur immeuble × chambres).
    cell_stats = lease_df.groupby(["lease_year"] + MIX_CELL)[price_col].agg(["mean", "size"])
    growth = {}
    for year in range(ANALYSIS_START_YEAR, FORECAST_YEAR):
        before = cell_stats.loc[year - 1] if year - 1 in cell_stats.index.get_level_values(0) else None
        after = cell_stats.loc[year]
        if before is None:
            continue
        common = before.index.intersection(after.index)
        weights = before.loc[common, "size"]
        change = after.loc[common, "mean"] / before.loc[common, "mean"] - 1
        growth[year] = (change * weights).sum() / weights.sum() * PERCENT
    return pd.Series(growth)


mix_effect = pd.DataFrame({"naïf (médiane annuelle)": naive.naive_yoy_pct,
                           "composition fixe": fixed_composition_growth_pct(leases)}).loc[ANALYSIS_START_YEAR:]
mix_effect["effet de mix (pts)"] = mix_effect.iloc[:, 0] - mix_effect.iloc[:, 1]
mix_effect.round(2)

**Lecture :**
- **2023 :** la médiane naïve monte de 10,8 %, mais à composition fixe la hausse n'est que d'environ 1,8 %. **Environ 9 points sont de l'effet de mix**, dû à l'arrivée du Carlyle et de The Met.
- **2025 :** l'effet s'inverse. La médiane naïve (+4,1 %) **sous-estime** la hausse à composition fixe (+6,9 %), parce que la part des baux dans les immeubles moins chers remonte.

L'effet de mix n'est donc pas un biais constant qu'on pourrait corriger une fois pour toutes. Une hausse mesurée sur des médianes annuelles décrit la stratégie de développement d'Équinoxe, pas l'évolution du prix d'un appartement.

## 12. Croissance à unité constante

Chaque bail est comparé au **bail précédent de la même unité** (`hUnit`, équivalent à `sPropCode + sUnitCode`). Les règles :
- **Filtre d'écart :** on garde seulement les écarts entre `MIN_GAP_YEARS` et `MAX_GAP_YEARS` ans.
- **Annualisation :** `(loyer / loyer précédent)^(1 / écart) − 1`, pour qu'un bail de 18 mois soit comparable à un bail de 12 mois.
- **Deux mesures :** on calcule la croissance pour le loyer contractuel **et** pour le loyer effectif.

In [ ]:
def annualized_growth_pct(current, previous, gap_years):
    return ((current / previous) ** (1 / gap_years) - 1) * PERCENT


def build_same_unit_pairs(lease_df, key=JOIN_KEY, min_gap=MIN_GAP_YEARS, max_gap=MAX_GAP_YEARS,
                          verbose=False):
    # Apparie chaque bail au bail précédent de la même unité et annualise la variation.
    key = [key] if isinstance(key, str) else list(key)
    df = lease_df.sort_values(key + ["lease_start"]).copy()
    by_unit = df.groupby(key)
    for col in ["rent_contract", "rent_effective", "rent_net", "lease_start", "discount_rate", "has_concession"]:
        df[f"prev_{col}"] = by_unit[col].shift(1)
    n_total = len(df)
    df = df.dropna(subset=["prev_lease_start"]).copy()   # copie explicite : aucun avertissement avec pandas 2.x
    n_with_prev = len(df)
    df["gap_years"] = (df.lease_start - df.prev_lease_start).dt.days / DAYS_PER_YEAR
    df = df[df.gap_years.between(min_gap, max_gap, inclusive="neither")]
    df = df[(df.prev_rent_contract > 0) & (df.rent_contract > 0)]
    df["growth_contract_pct"] = annualized_growth_pct(df.rent_contract, df.prev_rent_contract, df.gap_years)
    df["growth_effective_pct"] = annualized_growth_pct(df.rent_effective, df.prev_rent_effective, df.gap_years)
    df["growth_net_pct"] = annualized_growth_pct(df.rent_net, df.prev_rent_net, df.gap_years)
    if verbose:
        print(f"{n_total:,} baux -> {n_with_prev:,} avec un bail précédent "
              f"-> {len(df):,} paires comparables (écart dans ]{min_gap}, {max_gap}[ ans)")
    return df


pairs = build_same_unit_pairs(leases, verbose=True)

# Vérification : hUnit et sPropCode + sUnitCode donnent exactement les mêmes paires
pairs_unit_key = build_same_unit_pairs(leases, key=UNIT_KEY)
assert pairs[["hUnit", "lease_start"]].sort_values(["hUnit", "lease_start"]).reset_index(drop=True).equals(
       pairs_unit_key[["hUnit", "lease_start"]].sort_values(["hUnit", "lease_start"]).reset_index(drop=True))
print("OK : hUnit et sPropCode + sUnitCode donnent des paires identiques.")

# Le piège : sSite + sUnitCode fusionne des appartements différents
pairs_wrong = build_same_unit_pairs(leases, key=WRONG_UNIT_KEY)
print(f"Avec sSite + sUnitCode : {len(pairs_wrong):,} paires, dont des comparaisons entre appartements différents.")
print("Écart (jours) entre baux appariés :", pairs.gap_years.mul(DAYS_PER_YEAR).describe().round(0).to_dict())

In [ ]:
same_unit = (pairs[pairs.lease_year >= ANALYSIS_START_YEAR]
             .groupby("lease_year")
             .agg(pairs=("growth_contract_pct", "size"),
                  contract_median=("growth_contract_pct", "median"),
                  effective_median=("growth_effective_pct", "median")))
decomposition = pd.DataFrame({
    "naïf (médiane annuelle)": naive.naive_yoy_pct,
    "effet prix (même unité, contractuel)": same_unit.contract_median,
})
decomposition["effet mix (reste)"] = decomposition.iloc[:, 0] - decomposition.iloc[:, 1]

fig, ax = plt.subplots(figsize=(9, 4.5))
years = decomposition.dropna().index
ax.plot(years, decomposition.loc[years].iloc[:, 0], "o--", lw=2, label="naïf : médiane de tous les baux")
ax.plot(years, decomposition.loc[years].iloc[:, 1], "o-", lw=2, label="même unité, contractuel")
ax.plot(years, same_unit.loc[years, "effective_median"], "o-", lw=2, label="même unité, effectif")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("croissance annuelle (%)"); ax.set_title("La médiane naïve mesure surtout la composition")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
decomposition.dropna().round(2)

En 2023, la médiane naïve affiche +10,8 % alors que les mêmes unités n'ont augmenté que d'environ 2,3 %. **L'essentiel du « +10,8 % » est un effet de mix.**
La croissance à unité constante est la base de toute la suite.

## 13. Loyer contractuel vs loyer effectif

La croissance effective d'une paire se décompose exactement :

$$1 + g_{eff} = (1 + g_{contrat}) \times \frac{1 - d_{nouveau}}{1 - d_{précédent}}$$

où $d$ est le rabais (`discount_rate`) de chaque bail. Quand le rabais du nouveau bail dépasse celui de l'ancien, l'effectif croît moins vite que le contractuel.

In [ ]:
discount_path = (pairs[pairs.lease_year >= ANALYSIS_START_YEAR]
                 .groupby("lease_year")
                 .agg(contract_growth=("growth_contract_pct", "median"),
                      effective_growth=("growth_effective_pct", "median"),
                      discount_new=("discount_rate", "mean"),
                      discount_previous=("prev_discount_rate", "mean")))
discount_path["discount_change_pts"] = (discount_path.discount_new - discount_path.discount_previous) * PERCENT
discount_path["contract_minus_effective"] = discount_path.contract_growth - discount_path.effective_growth
discount_path.round(3)

Jusqu'en 2023, le rabais du nouveau bail est proche de celui du bail précédent : les deux croissances évoluent ensemble.
**À partir de 2024, le rabais moyen bondit : +1,8 point en 2024, puis +3,5 points en 2025.** Le moteur principal est PromoPay : un mois (presque) gratuit devient la norme. Le stationnement et le casier gratuits s'y ajoutent.
La croissance contractuelle (+7,0 % en 2025) surestime donc de plus de 3 points ce que le propriétaire encaisse réellement (+3,6 %).

Cela correspond à ce que décrit la SCHL pour 2025 : le marché se détend avec plus d'offre neuve, et les propriétaires offrent des mois gratuits **pour garder le loyer affiché élevé** plutôt que de le baisser.
C'est pour cette raison que notre chiffre principal est le loyer **effectif**.

### 13a. Les trois mesures de prix côte à côte

In [ ]:
PRICE_MEASURES = {"growth_contract_pct": "contractuel (sRent)", "growth_net_pct": "net des rabais de loyer",
                  "growth_effective_pct": "effectif (sRentEffective) - retenu"}
three_measures = (pairs[pairs.lease_year >= ANALYSIS_START_YEAR + 2].groupby("lease_year")
                  [list(PRICE_MEASURES)].median().rename(columns=PRICE_MEASURES))
ax = three_measures.plot(marker="o", lw=2, figsize=(9, 4))
ax.set_title("Croissance à unité constante (médiane) selon la mesure de prix"); ax.set_ylabel("%")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
three_measures.round(2)

Les trois mesures sont presque confondues jusqu'en 2023, puis s'écartent :
- **Le net** se place entre le contractuel et l'effectif. L'écart entre le net et l'effectif, c'est la part des concessions **accessoires** (surtout le stationnement offert), qui augmente nettement en 2024-2025.
- **Pour un propriétaire,** on présente le contractuel comme plafond, l'effectif comme ce qui est encaissé, et le net comme le prix de l'appartement.

### 13b. Piège : l'effet du bail précédent

La croissance effective dépend des concessions **des deux** baux de la paire. Si le bail précédent avait un rabais et le nouveau n'en a pas, la croissance effective est gonflée par un simple rattrapage. Dans le cas inverse, elle est écrasée.

In [ ]:
CONCESSION_FLAG = {0: "sans concession", 1: "avec concession"}
previous_lease_effect = (pairs[pairs.lease_year.isin(BACKTEST_YEARS[1:])]
                         .assign(previous=lambda d: d.prev_has_concession.map(CONCESSION_FLAG),
                                 new=lambda d: d.has_concession.map(CONCESSION_FLAG))
                         .groupby(["previous", "new"])
                         .agg(pairs=("growth_contract_pct", "size"),
                              contract_median=("growth_contract_pct", "median"),
                              effective_median=("growth_effective_pct", "median")))
previous_lease_effect.index.names = ["bail précédent", "nouveau bail"]
previous_lease_effect.round(2)

Ces écarts sont importants à l'échelle d'une paire, mais ils se compensent en partie sur un grand nombre de paires. C'est pour ça qu'on agrège par **médiane**.
C'est aussi pour ça que notre modèle de prévision traite explicitement **le rabais du bail précédent et celui du nouveau bail** (section 16), au lieu d'extrapoler directement la croissance effective.

## 14. Renouvellements vs relocations, et le cas de l'Ontario

- **Renouvellement (`is_renewal = 1`) :** un locataire en place. Au Québec, la hausse est encadrée par le TAL. En Ontario, elle est encadrée par la ligne directrice, **sauf exemption**.
- **Relocation (`is_renewal = 0`) :** l'unité change de locataire et se reloue au prix du marché.

**The Met (Ottawa).** Ses baux commencent en 2023, ce qui signifie une première occupation **après le 15 novembre 2018**.
Ces logements sont donc **exemptés de la ligne directrice ontarienne** : le propriétaire peut augmenter un locataire en place sans plafond, avec un préavis de 90 jours et au plus une fois par 12 mois.
On n'applique **ni** le plafond québécois **ni** le plafond ontarien à The Met.

**Québec, immeubles neufs.** Pour un immeuble de 5 ans ou moins, la section F du bail empêche le locataire de faire fixer le loyer par le TAL.
Le Carlyle et Westpark (mis en service en 2023-2024) sont dans ce cas. On vérifie dans les données s'ils se comportent différemment des immeubles anciens.

In [ ]:
first_lease_by_prop = leases.groupby("sPropCode").lease_start.min()
leases["first_lease_of_property"] = leases.sPropCode.map(first_lease_by_prop)
leases["ontario_guideline_exempt"] = ((leases.province == ONTARIO)
                                      & (leases.first_lease_of_property > ONTARIO_EXEMPTION_DATE))
# Âge déduit de la première année de bail de l'immeuble dans l'extrait (hypothèse à confirmer par Équinoxe)
first_year_by_building_all = leases.groupby("building").lease_year.min()
NEW_QC_BUILDINGS = set(first_year_by_building_all[
    first_year_by_building_all > FORECAST_YEAR - QC_SECTION_F_MAX_AGE_YEARS].index) - set(
    leases.loc[leases.province == ONTARIO, "building"])
print("Immeubles québécois de 5 ans ou moins en 2026 (section F) :", sorted(NEW_QC_BUILDINGS))
leases["quebec_section_f"] = leases.building.isin(NEW_QC_BUILDINGS)

def regulatory_regime(row):
    if row.province == ONTARIO:
        return "Ontario - exempté" if row.ontario_guideline_exempt else "Ontario - ligne directrice"
    return "Québec - section F (neuf)" if row.quebec_section_f else "Québec - TAL"

leases["regime"] = leases.apply(regulatory_regime, axis=1)
pairs = build_same_unit_pairs(leases)   # on reconstruit les paires avec les nouvelles colonnes
print(leases.groupby(["regime", "building"]).size().to_string())

In [ ]:
RENEWAL_LABEL = {0: "relocation", 1: "renouvellement"}
recent_pairs = pairs[pairs.lease_year >= ANALYSIS_START_YEAR + 3]
split = (recent_pairs.assign(kind=recent_pairs.is_renewal.map(RENEWAL_LABEL))
         .pivot_table(index="lease_year", columns=["province", "kind"],
                      values=["growth_contract_pct", "growth_effective_pct"], aggfunc="median"))
print("Croissance médiane à unité constante (%), par province et type :")
split.round(2)

In [ ]:
by_regime = (pairs[pairs.lease_year.isin(BACKTEST_YEARS[1:])]
             .assign(kind=lambda d: d.is_renewal.map(RENEWAL_LABEL))
             .pivot_table(index=["lease_year", "regime"], columns="kind",
                          values="growth_contract_pct", aggfunc="median"))
by_regime.round(2)

**Lecture :**
- Les **renouvellements menaient** jusqu'en 2023, quand le marché était tendu et que les nouvelles unités se louaient vite. Depuis 2024, ce sont **les relocations qui mènent**, de 2 à 2,5 points au-dessus des renouvellements.
- **The Met renouvelle à environ 6,3 % en 2025**, bien au-dessus de la ligne directrice ontarienne (2,5 %). C'est cohérent avec l'exemption post-2018 : ce n'est pas une infraction, c'est la règle qui s'applique à ce type d'immeuble.
- **Les immeubles section F se comportent comme les immeubles TAL.** On n'observe pas de prime de renouvellement nette au Carlyle ou à Westpark. On garde donc **un seul ancrage québécois (TAL)** pour les renouvellements. C'est plus simple, et c'est cohérent avec ce que la SCHL observe : beaucoup de propriétaires suivent la recommandation du TAL.

## 15. Données publiques et réconciliation avec la tendance interne

Le fichier `donnees_externes.csv` contient uniquement des données publiques, chacune avec sa source et son URL :
- **TAL :** pourcentage de base recommandé pour un logement non chauffé (2022-2026). La nouvelle méthode 2026 utilise la moyenne de l'IPC sur 3 ans.
- **Ontario :** ligne directrice d'augmentation (2022-2026).
- **Statistique Canada :** IPC annuel, tableau 18-10-0005-01, composante « loyer » et ensemble, Québec et Ontario.
- **SCHL :** Rapport sur le marché locatif, automne 2025 (Montréal et Ottawa). Croissance des loyers, unités relouées ou non, loyers de relocation des 2 chambres, inoccupation. L'inoccupation sert **uniquement de contexte de marché** : l'extrait ne permet pas de calculer celle du portefeuille.

**Calendrier de publication** (important pour un backtest honnête). La colonne `usage` du fichier le rend explicite : seules les séries `ancre_modele` entrent dans le modèle et le backtest. La SCHL (`contexte`) sert seulement à interpréter.
- Le taux TAL de l'année Y est publié **en janvier Y**, avant la quasi-totalité des renouvellements de Y, qui ont lieu surtout en juillet.
- La ligne directrice ontarienne de Y est publiée à l'été Y−1.
- L'IPC annuel de Y−1 est publié en janvier Y.

In [ ]:
external = pd.read_csv(EXTERNAL_DATA_FILE)
print("Séries par usage (ancre_modele = connue avant l'année prévue, utilisée au backtest ; contexte = interprétation seulement) :")
print(external.groupby(["usage", "series"]).year.agg(["min", "max", "size"]).to_string()); print()

def external_series(series, geo, frame=None):
    frame = external if frame is None else frame
    s = frame[(frame.series == series) & (frame.geo == geo)].set_index("year").value
    return s.sort_index()

tal_rate = external_series("tal_base_increase_pct", QUEBEC)
ontario_guideline = external_series("ontario_guideline_pct", ONTARIO)
cpi_rent_growth = {geo: external_series("cpi_rent_index", geo).pct_change() * PERCENT for geo in [QUEBEC, ONTARIO]}
cpi_all_growth = {geo: external_series("cpi_all_items_index", geo).pct_change() * PERCENT for geo in [QUEBEC, ONTARIO]}

external_view = pd.DataFrame({
    "TAL (%)": tal_rate, "Ontario ligne directrice (%)": ontario_guideline,
    "IPC loyer QC (%)": cpi_rent_growth[QUEBEC], "IPC loyer ON (%)": cpi_rent_growth[ONTARIO],
    "IPC ensemble QC (%)": cpi_all_growth[QUEBEC],
}).loc[ANALYSIS_START_YEAR:]
external_view.round(2)

In [ ]:
internal_renewal_qc = (pairs[(pairs.province == QUEBEC) & (pairs.is_renewal == 1)]
                       .groupby("lease_year").growth_contract_pct.median())
internal_turnover_qc = (pairs[(pairs.province == QUEBEC) & (pairs.is_renewal == 0)]
                        .groupby("lease_year").growth_contract_pct.median())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
yrs = external_view.index
axes[0].plot(yrs, internal_renewal_qc.reindex(yrs), "o-", lw=2, label="Équinoxe : renouvellements QC (même unité)")
axes[0].plot(tal_rate.index, tal_rate.values, "s--", lw=2, label="TAL : taux de base recommandé")
axes[0].plot(yrs, cpi_rent_growth[QUEBEC].reindex(yrs), "^:", lw=2, label="IPC loyer Québec")
axes[0].set_title("Renouvellements : la règle (TAL) ancre la tendance interne")
axes[0].set_ylabel("%"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

cmhc_turnover = external_series("cmhc_turnover_2bed_rent_change_pct", "Montreal CMA")
cmhc_nonturnover = external_series("cmhc_nonturnover_2bed_rent_change_pct", "Montreal CMA")
compare_cmhc = pd.DataFrame({
    "SCHL Montréal - relouées": cmhc_turnover, "Équinoxe QC - relocations": internal_turnover_qc.reindex(cmhc_turnover.index),
    "SCHL Montréal - non relouées": cmhc_nonturnover, "Équinoxe QC - renouvellements": internal_renewal_qc.reindex(cmhc_turnover.index)})
compare_cmhc.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Relocations : Équinoxe très loin de la moyenne SCHL")
axes[1].set_ylabel("%"); axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()
compare_cmhc.round(2)

**Réconciliation :**
1. **Renouvellements vs TAL.** Depuis que le TAL suit l'inflation (2023-2025), les renouvellements internes le suivent de près : 2,6 % vs 2,3 % ; 2,9 % vs 4,0 % ; 6,4 % vs 5,9 %.
   L'écart de 2022 (5,7 % vs 1,3 %) s'explique. Le TAL était alors loin sous le marché et l'inflation (IPC 2022 ≈ 6,7 %), et les immeubles récents ont pu s'en écarter.
   Le TAL 2026 de **3,1 %** est donc l'ancrage naturel des renouvellements québécois.
2. **Relocations vs SCHL.** La SCHL mesure environ +17 % pour une unité relouée à Montréal, contre environ +9 % chez Équinoxe.
   Ce n'est pas une contradiction. La SCHL capte le rattrapage d'unités anciennes louées très sous le marché (écart de 29 % entre loyers relouées et non relouées).
   Les unités d'Équinoxe sont récentes et **déjà au prix du marché** : il n'y a pas de rattrapage à faire. C'est pourquoi on n'utilise pas directement les chiffres SCHL de relocation, mais **l'écart interne** relocation − renouvellement.
3. **IPC loyer.** L'IPC loyer (QC +7,8 %, ON +4,3 % en 2025) mélange stock ancien et neuf. Il confirme la direction (forte hausse 2024-2025, décélération en Ontario) mais pas le niveau d'Équinoxe.
4. **Signal de marché pour 2026.** L'inoccupation monte : Montréal 2,9 %, Ottawa 3,0 % en 2025, et **6,7 % pour les logements construits après 2015 à Ottawa**. Les concessions ont donc toutes les raisons de rester élevées. C'est la principale source d'incertitude du chiffre effectif.

In [ ]:
# Les loyers affichés sont-ils un indicateur avancé ? On compare, au 31 décembre de Y-1,
# le dernier loyer affiché de chaque unité (après signature de son bail courant) au loyer de ce bail.
def asking_premium_signal(cutoff_year):
    cutoff = pd.Timestamp(f"{cutoff_year}-12-01")
    current = (leases[leases.sign_date <= cutoff].sort_values("lease_start")
               .groupby(JOIN_KEY).tail(1)[[JOIN_KEY, "rent_contract", "sign_date"]])
    obs = asking[asking.month_date <= cutoff].merge(current, on=JOIN_KEY)
    obs = obs[obs.month_date > obs.sign_date].sort_values("month_date").groupby(JOIN_KEY).tail(1)
    return ((obs.asking_rent / obs.rent_contract - 1) * PERCENT).median()

asking_check = pd.DataFrame({
    "prime affichée au 31/12 de Y-1 (%)": {y: asking_premium_signal(y - 1) for y in range(ANALYSIS_START_YEAR + 1, FORECAST_YEAR + 1)},
    "croissance contractuelle réalisée en Y (%)": same_unit.contract_median})
asking_check.round(2)

La prime du loyer affiché sur le loyer en cours reste **stable, autour de 2 %**, quelle que soit la hausse réalisée l'année suivante (2,3 % ou 7,0 %).
Les loyers affichés suivent le marché au lieu de l'anticiper. **On ne les utilise donc pas comme variable de prévision.** On les garde comme contrôle de cohérence : la prime de fin 2025 est d'environ 1,8 %, ce qui confirme un marché qui ralentit.

## 16. Modèle de prévision

### Méthode : modèle par composantes ancré sur les règles

On prévoit séparément chaque bloc, puis on agrège. Dans les formules, Y est l'année prévue.

| Bloc | Prévision du loyer **contractuel** | Justification |
|---|---|---|
| Renouvellement Québec | taux TAL de Y | La règle ancre le comportement (section 15). |
| Renouvellement Ontario (The Met, exempté) | IPC loyer Ontario de Y−1 | Pas de plafond : la hausse suit le marché ontarien. On n'applique **pas** la ligne directrice. |
| Relocation (les deux provinces) | ancrage de renouvellement + écart moyen relocation − renouvellement sur `GAP_WINDOW_YEARS` ans | Le marché se mesure *par rapport* à la hausse des locataires en place. |

**Passage au loyer effectif :** $1+g_{eff} = (1+g_{contrat}) \cdot \frac{1-d_{nouveau}}{1-d_{précédent}}$.
- $d_{précédent}$ est le rabais moyen des baux de Y−1. Ce sont eux qui deviendront « bail précédent » en Y, et ce rabais est connu au 31/12/Y−1.
- $d_{nouveau} = d_{précédent} + \phi \times$ (variation annuelle moyenne du rabais sur les 2 dernières années). Ici $\phi \in \{0 ; 0{,}5 ; 1\}$ : persistance, tendance amortie ou tendance pleine. **Le choix de $\phi$ est fait par backtest.**

**Agrégation :** moyenne pondérée des blocs, avec la part observée de chaque bloc (province × type) parmi les paires de Y−1.

**Pourquoi pas XGBoost ?**
- On a 7 années de paires, dont 3 de backtest, et un seul « choc » annuel commun à toutes les unités : un modèle d'arbres apprendrait l'année, pas le mécanisme.
- Le modèle par composantes a **3 paramètres interprétables** : écart relocation, rabais, poids. Il utilise directement l'information réglementaire connue d'avance.

On le compare à deux modèles de référence : **naïf** (la hausse effective de Y−1) et **moyenne sur 3 ans**.

In [ ]:
def renewal_anchor_pct(province, target_year, external_tables):
    # Hausse contractuelle de renouvellement attendue selon la règle applicable.
    if province == QUEBEC:
        return external_tables["tal_rate"].loc[target_year]
    # Ontario, The Met : exempté de la ligne directrice -> suit le marché (IPC loyer Ontario, Y-1)
    return external_tables["cpi_rent_growth"][ONTARIO].loc[target_year - 1]


def fit_components(lease_df, target_year, damping, discount_col="discount_rate"):
    # Paramètres estimés uniquement avec l'information disponible au 31/12 de target_year - 1.
    history = lease_df[lease_df.sign_date < pd.Timestamp(f"{target_year}-01-01")]
    history_pairs = build_same_unit_pairs(history)
    last_year = target_year - 1
    yearly = history_pairs.groupby(["lease_year", "is_renewal"]).growth_contract_pct.median().unstack()
    gap_years = yearly.loc[last_year - GAP_WINDOW_YEARS + 1:last_year]
    turnover_gap = (gap_years[0] - gap_years[1]).mean()

    discount_by_year = history.groupby("lease_year")[discount_col].mean()
    discount_previous = discount_by_year.loc[last_year]
    discount_trend = ((discount_by_year.loc[last_year] - discount_by_year.loc[last_year - DISCOUNT_TREND_YEARS])
                      / DISCOUNT_TREND_YEARS)
    discount_new = discount_previous + damping * discount_trend

    last_pairs = history_pairs[history_pairs.lease_year == last_year]
    weights = last_pairs.groupby(["province", "is_renewal"]).size()
    weights = weights / weights.sum()
    return dict(turnover_gap=turnover_gap, discount_previous=discount_previous,
                discount_new=discount_new, weights=weights, history_pairs=history_pairs)


def effective_from_contract(contract_pct, discount_new, discount_previous):
    return ((1 + contract_pct / PERCENT) * (1 - discount_new) / (1 - discount_previous) - 1) * PERCENT


def forecast_segments(params, target_year, external_tables):
    rows = []
    for (province, is_renewal), weight in params["weights"].items():
        anchor = renewal_anchor_pct(province, target_year, external_tables)
        contract = anchor if is_renewal == 1 else anchor + params["turnover_gap"]
        effective = effective_from_contract(contract, params["discount_new"], params["discount_previous"])
        rows.append(dict(province=province, kind=RENEWAL_LABEL[is_renewal], weight=weight,
                         contract_pct=contract, effective_pct=effective))
    return pd.DataFrame(rows)


EXTERNAL_TABLES = {"tal_rate": tal_rate, "ontario_guideline": ontario_guideline,
                   "cpi_rent_growth": cpi_rent_growth}

### `estimate_2026()` et `backtest()`

`backtest(leases, target_year)` « se place » au 31 décembre de `target_year − 1` :
- **Données utilisées :** seulement les baux signés avant cette date, ainsi que les paramètres publics publiés à temps.
- **Prévision :** elle produit ensuite une prévision de `target_year`.
- **Comparaison :** elle compare cette prévision à la hausse **réellement mesurée** en `target_year`, avec **exactement la même agrégation** que la prévision : médiane de chaque segment (province × type de bail), pondérée par le poids réel du segment cette année-là. Les modèles de référence utilisent la même mesure, pour comparer des chiffres comparables.

In [ ]:
SEGMENT_COLS = ["province", "is_renewal"]

def predict_year(lease_df, target_year, damping, external_tables=EXTERNAL_TABLES, discount_col="discount_rate"):
    params = fit_components(lease_df, target_year, damping, discount_col)
    segments = forecast_segments(params, target_year, external_tables)
    return dict(effective_pct=(segments.weight * segments.effective_pct).sum(),
                contract_pct=(segments.weight * segments.contract_pct).sum(),
                segments=segments, params=params)


def realized_growth(pair_df, year, growth_col="growth_effective_pct"):
    # Hausse mesurée selon notre définition : médiane par segment, pondérée par le poids réel du segment.
    year_pairs = pair_df[pair_df.lease_year == year]
    by_segment = year_pairs.groupby(SEGMENT_COLS)[growth_col].agg(["median", "size"])
    return (by_segment["median"] * by_segment["size"]).sum() / by_segment["size"].sum()


def baseline_predictions(lease_df, target_year):
    history_pairs = build_same_unit_pairs(lease_df[lease_df.sign_date < pd.Timestamp(f"{target_year}-01-01")])
    past = pd.Series({y: realized_growth(history_pairs, y)
                      for y in range(target_year - BASELINE_WINDOW_YEARS, target_year)})
    return {"naïf (Y-1)": past.loc[target_year - 1], "moyenne 3 ans": past.mean()}


def actual_growth(lease_df, target_year):
    all_pairs = build_same_unit_pairs(lease_df)
    return dict(effective_pct=realized_growth(all_pairs, target_year, "growth_effective_pct"),
                contract_pct=realized_growth(all_pairs, target_year, "growth_contract_pct"),
                net_pct=realized_growth(all_pairs, target_year, "growth_net_pct"))


def backtest(leases, target_year, damping=None):
    # Exécute la méthode comme au 31/12 de target_year - 1 et compare avec la réalité de target_year.
    dampings = CONCESSION_DAMPING if damping is None else {"choisi": damping}
    actual = actual_growth(leases, target_year)
    row = {"année": target_year, "réel effectif": actual["effective_pct"], "réel contractuel": actual["contract_pct"]}
    for label, phi in dampings.items():
        pred = predict_year(leases, target_year, phi)
        row[f"composantes - {label}"] = pred["effective_pct"]
        row["composantes - contractuel"] = pred["contract_pct"]
    row.update(baseline_predictions(leases, target_year))
    return row


backtest_results = pd.DataFrame([backtest(leases, y) for y in BACKTEST_YEARS]).set_index("année")
backtest_results.round(2)

In [ ]:
prediction_cols = [c for c in backtest_results.columns if not c.startswith("réel")]
errors = pd.DataFrame({
    col: backtest_results[col] - backtest_results["réel contractuel" if "contractuel" in col else "réel effectif"]
    for col in prediction_cols})
scores = pd.DataFrame({"MAE (pts)": errors.abs().mean(), "biais moyen (pts)": errors.mean(),
                       "erreur max (pts)": errors.abs().max()}).sort_values("MAE (pts)")
print("Erreur = prévision - réel ; MAE = erreur absolue moyenne sur 2023-2025")
scores.round(2)

In [ ]:
effective_candidates = {label: f"composantes - {label}" for label in CONCESSION_DAMPING}
chosen_label = min(effective_candidates, key=lambda lbl: scores.loc[effective_candidates[lbl], "MAE (pts)"])
CHOSEN_DAMPING = CONCESSION_DAMPING[chosen_label]
print(f"Hypothèse de concessions retenue par backtest : « {chosen_label} » (phi = {CHOSEN_DAMPING})")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(backtest_results.index, backtest_results["réel effectif"], "ko-", lw=3, label="réel (effectif)")
for col in prediction_cols:
    if "contractuel" not in col:
        ax.plot(backtest_results.index, backtest_results[col], "o--", label=col)
ax.set_xticks(list(BACKTEST_YEARS)); ax.set_ylabel("croissance effective à unité constante (%)")
ax.set_title("Backtest 2023-2025 : prévision faite au 31 décembre de l'année précédente")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Le choix de φ est-il fragile ?** On trace l'erreur moyenne du backtest sur une grille de φ qui va au-delà de 1. Au-delà de 1, on suppose que la hausse des concessions **s'accélère**.

In [ ]:
actual_effective = backtest_results["réel effectif"]
phi_sensitivity = pd.Series({
    phi: np.mean([abs(predict_year(leases, y, phi)["effective_pct"] - actual_effective.loc[y]) for y in BACKTEST_YEARS])
    for phi in PHI_GRID})
ax = phi_sensitivity.plot(marker="o", figsize=(8, 3.8))
ax.axvline(CHOSEN_DAMPING, color="red", ls="--", label=f"φ retenu = {CHOSEN_DAMPING}")
ax.set_xlabel("φ (part de la tendance des concessions prolongée)"); ax.set_ylabel("MAE backtest (pts)")
ax.set_title("Sensibilité de l'erreur au paramètre de concessions"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
phi_sensitivity.round(2).rename("MAE (pts)").to_frame().T

**Lecture honnête :** l'erreur diminue **continuellement** quand φ augmente, même au-delà de 1.
Sur 2023-2025, les concessions ont donc progressé **encore plus vite** qu'une simple prolongation de tendance. Le saut de 2025 (+3,5 points de rabais) dépasse celui de 2024 (+1,8).

On **plafonne quand même φ à 1**, et ce choix est fait *a priori*, pas sur le backtest :
- Un φ supérieur à 1 suppose que l'accélération continue. Le rabais moyen dépasserait alors 13 % du loyer en 2026, soit plus d'un mois et demi gratuit par an en moyenne.
- Les concessions ne peuvent pas croître indéfiniment. La SCHL note que l'offre neuve, qui les alimente, est déjà en cours d'absorption.
- Choisir φ = 1,5 parce qu'il minimise l'erreur sur trois points serait du sur-ajustement.

**Conséquence :** notre scénario central (φ = 1) est déjà un scénario **prudent mais non extrême**, et le risque résiduel est à la baisse. Le scénario « persistance » (φ = 0) reste le plafond, si Équinoxe arrête d'augmenter ses concessions.

**Lecture du backtest :**
- **Volet contractuel :** l'ancrage TAL + écart de relocation a une erreur moyenne d'environ 0,7 point. Il **sous-estime** légèrement, surtout en 2025, où les relocations ont dépassé l'écart historique.
- **Volet effectif :** le modèle par composantes avec tendance des concessions est **le meilleur des six**, avec une erreur moyenne inférieure à 1 point. La moyenne sur 3 ans fait 1,4 point et le naïf 1,55. Son erreur vient surtout des concessions, dont la hausse brutale en 2024-2025 n'était pas visible avant.
- **Modèles de référence :** le naïf et la moyenne sur 3 ans ne savent pas que le TAL passe de 5,9 % à 3,1 % en 2026. Ils prolongeraient mécaniquement 2025.

Avec trois années de backtest, l'écart de score entre méthodes n'est pas statistiquement solide. On retient le modèle par composantes pour trois raisons :
- il est **le mieux justifié** (règles + mécanisme des concessions) ;
- il **gagne** le backtest ;
- il est le seul à intégrer le **changement de régime de 2026** (TAL à 3,1 %), connu d'avance.

## 17. Prévision 2026

Les intrants 2026 :
- **Québec :** TAL 2026 = **3,1 %** (nouvelle méthode, moyenne IPC sur 3 ans).
- **Ontario :** The Met reste exempté ; l'ancrage est l'IPC loyer Ontario 2025 = **+4,3 %**. À titre de comparaison seulement, la ligne directrice 2026 est de 2,1 %.
- **Relocation et concessions :** écart relocation − renouvellement et rabais mesurés jusqu'au 31/12/2025.

In [ ]:
def external_tables_from_frame(frame):
    # Tables d'ancrage (TAL, ligne directrice, IPC loyer) construites à partir de donnees_externes.csv.
    return {"tal_rate": external_series("tal_base_increase_pct", QUEBEC, frame),
            "ontario_guideline": external_series("ontario_guideline_pct", ONTARIO, frame),
            "cpi_rent_growth": {geo: external_series("cpi_rent_index", geo, frame).pct_change() * PERCENT
                                for geo in [QUEBEC, ONTARIO]}}


def estimate_2026(leases, asking, external=None, damping=None):
    # Retourne la hausse 2026 en %, selon notre définition :
    # croissance médiane du loyer EFFECTIF à unité constante, renouvellements + relocations pondérés.
    # `asking` est accepté pour garder la signature du gabarit, mais n'entre pas dans le modèle (section 15).
    # `external` : None (tables de la section 15), le DataFrame de donnees_externes.csv, ou un dict de tables.
    if external is None:
        external_tables = EXTERNAL_TABLES
    elif isinstance(external, pd.DataFrame):
        external_tables = external_tables_from_frame(external)
    else:
        external_tables = external
    phi = CHOSEN_DAMPING if damping is None else damping
    return predict_year(leases, FORECAST_YEAR, phi, external_tables)["effective_pct"]


# Même résultat si on passe le DataFrame externe, comme le suggère la signature du gabarit
assert np.isclose(estimate_2026(leases, asking, external), estimate_2026(leases, asking))

forecast_central = predict_year(leases, FORECAST_YEAR, CHOSEN_DAMPING)
print(f"Hausse 2026 (effective, unité constante) : {estimate_2026(leases, asking):.2f} %")
print(f"Hausse 2026 (contractuelle, unité constante) : {forecast_central['contract_pct']:.2f} %")
params_2026 = forecast_central["params"]
print(f"Écart relocation - renouvellement : {params_2026['turnover_gap']:.2f} pts")
print(f"Rabais précédent -> nouveau : {params_2026['discount_previous']:.1%} -> {params_2026['discount_new']:.1%}")
forecast_central["segments"].round(2)

**Sensibilité à la mesure de prix.** On applique la même méthode au loyer **net des seuls rabais de loyer** : le rabais modélisé ne comprend alors que PromoPay et freerent.
On la backteste aussi, pour vérifier que la conclusion ne dépend pas du traitement du stationnement offert.

In [ ]:
NET_DISCOUNT_COL = "net_discount_rate"
forecast_net = predict_year(leases, FORECAST_YEAR, CHOSEN_DAMPING, discount_col=NET_DISCOUNT_COL)
net_backtest = pd.DataFrame({
    y: {"prévu (net)": predict_year(leases, y, CHOSEN_DAMPING, discount_col=NET_DISCOUNT_COL)["effective_pct"],
        "réel (net)": actual_growth(leases, y)["net_pct"]} for y in BACKTEST_YEARS}).T
net_mae = (net_backtest["prévu (net)"] - net_backtest["réel (net)"]).abs().mean()
print(f"Hausse 2026 - loyer net des rabais de loyer : {forecast_net['effective_pct']:.2f} % "
      f"(MAE backtest : {net_mae:.2f} pt)")
net_backtest.round(2)

### 17a. Scénarios et intervalle

- **Scénarios.** On fait tourner le modèle avec les trois hypothèses de concessions. **Central** = l'hypothèse retenue par le backtest. **Haut** = la plus favorable : les rabais se stabilisent au niveau de 2025. **Bas** = la moins favorable. Si le backtest retient l'hypothèse la plus prudente, central et bas coïncident, et on le dit.
- **Intervalle bootstrap.** On rééchantillonne les **unités** (`hUnit`) avec remise et on réestime le modèle central. Cela mesure l'incertitude d'échantillonnage sur l'écart de relocation, les rabais et les poids.
- **Intervalle empirique.** Central ± l'erreur absolue moyenne du modèle retenu au backtest. Il ne suppose aucune loi statistique, ce que trois années de test ne permettraient pas de vérifier. C'est la mesure de prudence la plus honnête.

In [ ]:
scenarios = pd.DataFrame({
    label: {"φ (concessions)": phi,
            "effectif 2026 (%)": predict_year(leases, FORECAST_YEAR, phi)["effective_pct"],
            "contractuel 2026 (%)": predict_year(leases, FORECAST_YEAR, phi)["contract_pct"]}
    for label, phi in CONCESSION_DAMPING.items()}).T
scenarios["retenu par backtest"] = scenarios.index == chosen_label
scenario_summary = {"bas": scenarios["effectif 2026 (%)"].min(),
                    "central": scenarios.loc[chosen_label, "effectif 2026 (%)"],
                    "haut": scenarios["effectif 2026 (%)"].max()}

rng = np.random.default_rng(RANDOM_SEED)
rows_by_unit = list(leases.groupby(JOIN_KEY).indices.values())   # positions des baux de chaque unité
bootstrap_estimates = []
for draw in range(N_BOOTSTRAP):
    picked = rng.integers(0, len(rows_by_unit), size=len(rows_by_unit))
    positions = np.concatenate([rows_by_unit[i] for i in picked])
    # chaque tirage reçoit un identifiant neuf pour qu'une unité tirée deux fois ne s'apparie pas avec elle-même
    new_ids = np.repeat(np.arange(len(picked)), [len(rows_by_unit[i]) for i in picked])
    sample = leases.iloc[positions].assign(hUnit=new_ids)
    bootstrap_estimates.append(predict_year(sample, FORECAST_YEAR, CHOSEN_DAMPING)["effective_pct"])
ci_low, ci_high = np.percentile(bootstrap_estimates, BOOTSTRAP_CI)
print(f"Intervalle bootstrap {BOOTSTRAP_CI[1] - BOOTSTRAP_CI[0]:.0f} % (unités rééchantillonnées) : "
      f"[{ci_low:.2f} % ; {ci_high:.2f} %]")
chosen_mae = scores.loc[effective_candidates[chosen_label], "MAE (pts)"]
empirical_low = scenario_summary["central"] - chosen_mae
empirical_high = scenario_summary["central"] + chosen_mae
print(f"Intervalle empirique (central ± MAE backtest {chosen_mae:.2f}) : [{empirical_low:.2f} % ; {empirical_high:.2f} %]")
scenarios.round(2)

Le bootstrap ne capte que le bruit d'échantillonnage : l'intervalle est étroit.
**L'incertitude dominante est l'hypothèse de concessions**, d'où l'importance des scénarios. Un gestionnaire peut agir directement sur ce levier.
Chaque point de rabais moyen en moins sur les nouveaux baux ajoute environ un point à la hausse effective.

### 17b. Prévision par immeuble et par nombre de chambres (bonus)

On applique exactement la même formule à chaque groupe :
- **Propres au groupe :** l'ancrage de sa province, son poids renouvellement/relocation et son rabais de 2025.
- **Communs à tout le portefeuille :** l'écart de relocation et la tendance du rabais. On les garde au niveau portefeuille pour ne pas surajuster sur de petits échantillons.

In [ ]:
def forecast_by_group(lease_df, group_col, target_year=FORECAST_YEAR, damping=None):
    phi = CHOSEN_DAMPING if damping is None else damping
    portfolio = fit_components(lease_df, target_year, phi)
    last_year = target_year - 1
    discount_shift = portfolio["discount_new"] - portfolio["discount_previous"]
    rows = []
    history = lease_df[lease_df.sign_date < pd.Timestamp(f"{target_year}-01-01")]
    last_pairs = portfolio["history_pairs"].query("lease_year == @last_year")
    for group, group_pairs in last_pairs.groupby(group_col):
        # Un groupe (ex. 2 chambres) peut couvrir les deux provinces : on pondère chaque bloc.
        block_weights = group_pairs.groupby(["province", "is_renewal"]).size() / len(group_pairs)
        contract = sum(w * (renewal_anchor_pct(p, target_year, EXTERNAL_TABLES)
                            + (0 if r == 1 else portfolio["turnover_gap"]))
                       for (p, r), w in block_weights.items())
        discount_prev = history[(history[group_col] == group) & (history.lease_year == last_year)].discount_rate.mean()
        effective = effective_from_contract(contract, discount_prev + discount_shift, discount_prev)
        rows.append({group_col: group, "province(s)": "/".join(sorted(group_pairs.province.unique())),
                     "paires 2025": len(group_pairs), "part renouvellements": group_pairs.is_renewal.mean(),
                     "rabais 2025 (%)": discount_prev * PERCENT,
                     "contractuel 2026 (%)": contract, "effectif 2026 (%)": effective})
    return pd.DataFrame(rows).set_index(group_col)


by_building = forecast_by_group(leases, "building")
building_scenarios = pd.concat({label: forecast_by_group(leases, "building", damping=phi)["effectif 2026 (%)"]
                                for label, phi in CONCESSION_DAMPING.items()}, axis=1)
by_building["effectif bas (%)"] = building_scenarios.min(axis=1)
by_building["effectif haut (%)"] = building_scenarios.max(axis=1)
by_beds = forecast_by_group(leases, "beds")
display(by_building.round(2))
display(by_beds.round(2))

In [ ]:
# Petit tableau de bord de la prévision par immeuble
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [3, 2]})
order = by_building.sort_values("effectif 2026 (%)").index
central = by_building.loc[order, "effectif 2026 (%)"]
low = by_building.loc[order, "effectif bas (%)"]; high = by_building.loc[order, "effectif haut (%)"]
axes[0].barh(order, by_building.loc[order, "contractuel 2026 (%)"], color="#d9d9d9", label="contractuel")
axes[0].errorbar(central, order, xerr=[central - low, high - central], fmt="o", color="#1f4e79",
                 capsize=4, label="effectif (central, bas-haut)")
axes[0].set_xlabel("hausse 2026 (%)"); axes[0].set_title("Prévision 2026 par immeuble")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, axis="x")

scen = scenarios["effectif 2026 (%)"]
axes[1].bar(scen.index, scen.values, color=["#1f4e79" if lbl == chosen_label else "#a6a6a6" for lbl in scen.index])
axes[1].axhline(forecast_central["contract_pct"], color="grey", ls="--", label="contractuel")
axes[1].set_title("Portefeuille : scénarios de concessions"); axes[1].set_ylabel("%")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# Sauvegarde des paramètres du modèle : agrégats seulement, aucune ligne CRM.
model_params = {
    "definition": "croissance médiane annualisée du loyer effectif à unité constante (hUnit), "
                  "renouvellements + relocations pondérés",
    "forecast_year": FORECAST_YEAR,
    "concession_hypothesis": chosen_label, "damping_phi": CHOSEN_DAMPING,
    "turnover_gap_pts": round(float(params_2026["turnover_gap"]), 3),
    "discount_previous": round(float(params_2026["discount_previous"]), 4),
    "discount_new": round(float(params_2026["discount_new"]), 4),
    "weights": {f"{p}|{RENEWAL_LABEL[r]}": round(float(w), 4) for (p, r), w in params_2026["weights"].items()},
    "anchors_pct": {"TAL_2026": float(tal_rate.loc[FORECAST_YEAR]),
                    "Ontario_CPI_rent_2025": round(float(cpi_rent_growth[ONTARIO].loc[FORECAST_YEAR - 1]), 3)},
    "estimate_effective_pct": round(float(forecast_central["effective_pct"]), 2),
    "estimate_contract_pct": round(float(forecast_central["contract_pct"]), 2),
    "scenarios_effective_pct": {k: round(float(v), 2) for k, v in scenario_summary.items()},
    "concession_hypotheses_effective_pct": {k: round(float(v), 2) for k, v in scenarios["effectif 2026 (%)"].items()},
    "bootstrap_ci_pct": [round(float(ci_low), 2), round(float(ci_high), 2)],
    "empirical_interval_pct": [round(float(empirical_low), 2), round(float(empirical_high), 2)],
    "net_of_rent_concessions_pct": round(float(forecast_net["effective_pct"]), 2),
    "backtest_mae_pts": {k: round(float(v), 2) for k, v in scores["MAE (pts)"].items()},
}
MODEL_PARAMS_FILE.write_text(json.dumps(model_params, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(model_params, indent=2, ensure_ascii=False))

## 18. Résultat, hypothèses et limites

### Résultat

In [ ]:
final_summary = pd.DataFrame({
    "valeur (%)": [scenario_summary["central"], empirical_low, empirical_high,
                   scenario_summary["bas"], scenario_summary["haut"], ci_low, ci_high,
                   forecast_central["contract_pct"], forecast_net["effective_pct"]],
}, index=["Hausse 2026 - effective, unité constante (CENTRAL)",
          "  intervalle empirique - bas (central - MAE)", "  intervalle empirique - haut (central + MAE)",
          "  scénario de concessions le plus bas", "  scénario de concessions le plus haut",
          "  bootstrap borne basse", "  bootstrap borne haute",
          "Hausse 2026 - contractuelle (plafond)", "Hausse 2026 - nette des seuls rabais de loyer"])
print(f"Définition : croissance médiane annualisée du loyer effectif à unité constante (hUnit),\n"
      f"renouvellements et relocations pondérés. Hypothèse de concessions : « {chosen_label} ».")
final_summary.round(2)

**Comment lire ce résultat :**
- **La hausse contractuelle** est ce que les baux 2026 afficheront. Elle est tirée surtout par le TAL 2026 de 3,1 %, plus bas qu'en 2025, et par l'écart des relocations.
- **La hausse effective** est ce qui sera encaissé. Elle dépend surtout de l'évolution des concessions.
- **L'écart entre les deux** est le coût des concessions. C'est le levier de gestion principal pour 2026 : chaque point de rabais moyen évité sur les nouveaux baux rapporte environ un point de hausse effective.
- **La mesure nette des seuls rabais de loyer** se situe entre les deux. Elle exclut le stationnement et le casier offerts. Elle montre qu'une bonne partie de l'écart vient des concessions accessoires, sur lesquelles la gestion peut agir sans toucher au loyer.

**Recommandation de budget.** Budgéter les revenus sur la hausse **effective centrale**, et fixer les loyers de renouvellement sur la hausse **contractuelle**. Le scénario « persistance » montre ce qu'on gagne si les concessions cessent de croître en 2026.

### Hypothèses et limites

**Hypothèses :**
1. **`sRentEffective` est le revenu net perçu par unité.** C'est le loyer contractuel moins toutes les concessions du bail, étalées sur sa durée (vérifié par reconstruction, à moins de 1 $ près en médiane). PromoPay est un rabais unique, et non un montant mensuel.
2. **`hUnit` est la clé d'unité.** Elle est strictement équivalente à `sPropCode + sUnitCode` (vérifié). Les paires sont limitées aux écarts de ]0,5 ; 2,5[ ans et annualisées.
3. **Le TAL ancre les renouvellements québécois,** y compris dans les immeubles section F, qui se comportent comme les autres dans les données. Le taux TAL de Y est connu avant les renouvellements de Y.
4. **The Met est exempté de la ligne directrice ontarienne** (première occupation après le 15 novembre 2018). Ses renouvellements suivent le marché, approché par l'IPC loyer de l'Ontario. On n'applique jamais le plafond d'une province à l'autre.
5. **L'écart relocation − renouvellement des 3 dernières années se maintient.**
6. **Le mix 2026 ressemble à celui de 2025,** en poids de renouvellements et relocations par province.

**Limites :**
- **Pas d'occupation ni d'inoccupation.** L'extrait est un échantillon fixe d'unités ; ces indicateurs ne peuvent pas être calculés et ne sont pas cités pour le portefeuille.
- **Seulement trois années de backtest,** dont deux avec un changement de régime des concessions. L'erreur passée du volet effectif (environ 1 point) est la bonne mesure de prudence.
- **Peu d'historique pour The Met** (2 années de paires). Sa prévision repose davantage sur l'ancrage externe.
- **Taux TAL 2022 et 2023 :** on utilise les taux « logement non chauffé » rapportés par les médias, car le TAL publiait alors des pourcentages par composante.
- **Aucune information publiée après décembre 2025** n'est utilisée, sauf le taux TAL 2026, publié en janvier 2026 et nécessairement connu avant les renouvellements.
- **Âge des immeubles :** le statut section F et l'exemption ontarienne sont déduits de la date du premier bail dans l'extrait. La date réelle de première occupation devrait être confirmée par Équinoxe.

### Respect des règles du défi

| Règle | Comment elle est respectée |
|---|---|
| Python et Jupyter, à partir du notebook de départ | Les sections 1 à 6 de `starter.ipynb` sont reprises telles quelles en tête. Nos sections commencent à la section 7. |
| Fichiers sources intacts | Les CSV sont seulement lus. Tout renommage se fait sur des copies (`leases`, `concessions`, `asking`, `listings`). |
| Aucune donnée CRM dans les livrables | Aucune ligne brute n'est affichée (seulement des agrégats et des schémas). `model_params.json` ne contient que des paramètres agrégés. |
| Clé `sPropCode + sUnitCode` (ou `hUnit`), jamais `sSite + sUnitCode` | Jointures sur `hUnit`. Une assertion vérifie que les paires sont identiques avec `sPropCode + sUnitCode`. `sSite` n'apparaît que pour illustrer le piège. |
| Pas de nombre magique | Tous les paramètres sont des constantes nommées (section 7). |
| Markdown à chaque étape qui modifie les données | Chaque transformation est précédée d'une cellule explicative. |
| Renouvellements vs relocations | Séparés à l'analyse (section 14) et prévus séparément (section 16). |
| Règles Québec vs Ontario | TAL au Québec. Exemption post-2018 pour The Met, qui n'est jamais plafonné par l'une ou l'autre province. |
| Pas d'occupation tirée de l'extrait | Seule l'inoccupation publiée par la SCHL est citée, comme contexte de marché. |
| Sources et IA citées | `donnees_externes.csv` (une source par ligne) et section Références. |

## 19. Références

**Données publiques** (détail et URL dans `donnees_externes.csv`) :
- Tribunal administratif du logement : calcul annuel de l'augmentation des loyers ; communiqués 2022-2023 ; taux 2026 de 3,1 % (Radio-Canada, Le Devoir, janvier 2026). https://www.tal.gouv.qc.ca
- Gouvernement de l'Ontario : ligne directrice sur l'augmentation des loyers (2,1 % en 2026) et exemption des logements occupés pour la première fois après le 15 novembre 2018. https://www.ontario.ca/page/rent-increase-guideline
- Statistique Canada : tableau 18-10-0005-01, Indice des prix à la consommation, moyenne annuelle (composante « loyer », « ensemble »). https://www150.statcan.gc.ca
- SCHL : Rapport sur le marché locatif, automne 2025 (Montréal, Ottawa). https://www.cmhc-schl.gc.ca

**Outils d'IA :**
- Claude (Anthropic), via Claude Code, a servi à explorer les données, rédiger le code et le texte de ce notebook, et rechercher les sources publiques. Toutes les sorties ont été exécutées et vérifiées dans le notebook.

**Méthode :**
- Gardner, E. S. & McKenzie, E. (1985). *Forecasting trends in time series* : tendance amortie, utilisée pour l'hypothèse de concessions.